# Initial Datahandeling

### This notebook shows the initial datahandling of utterences in the danihs Parliament.
### It includes the following:

- Loading the two overlapping datasets of utterences in the danish parliament
    - Due to the ParlLawSpeech data being in .rds format, it was read into r, converted to .csv format and then saved in this repo
- Merging them by end data of corp_v2
- Sentence segmentation
- Reformatting to json object for easier handling during labeling for validation, training and inference
- Datacleaning
- Splitting into training data (including validationset to be extracted), and inference data

In [ ]:
# Load packages

import pandas as pd
import os


# Corp_Folketing_V2

In [2]:
#read the corp folketing 

corp_v2_path = os.path.join("..",
                            "..",
                            "raw_data",
                            "Corp_Folketing_V2.csv")

# Read the two datasets
df_corp = pd.read_csv(corp_v2_path)
x = df_corp.pop("Unnamed: 0") #dummy variable

#make date a datetime object
df_corp["date"] = pd.to_datetime(df_corp["date"])

In [9]:
#This dataset spans the following time period

min_time_corp = df_corp["date"].min()
max_time_corp = df_corp["date"].max()

print(f"The Corp dataset has coverage from {min_time_corp.date()} to {max_time_corp.date()}")

The Corp dataset has coverage from 1997-10-07 to 2018-12-20


# PLS speeches

In [ ]:
#read the PLS data. 
# OBS has been made to .csv in file ./PLS_rds_to_csv.RMD

PLS_path = os.path.join("..",
                            "..",
                            "..",
                            "..",
                            "Data_outside_git",
                            "PLS_speeches.csv")

# Read the two datasets
df_PLS = pd.read_csv(PLS_path)

#make date a datetime object
df_PLS["date"] = pd.to_datetime(df_PLS["date"])

In [10]:
#This dataset spans the following time period

min_time_PLS = df_PLS["date"].min()
max_time_PLS = df_PLS["date"].max()

print(f"The PLS dataset has coverage from {min_time_PLS.date()} to {max_time_PLS.date()}")

The PLS dataset has coverage from 2007-11-27 to 2022-10-06


# Merging datasets

In [6]:
# 

# Keep only rows in df2 strictly after max date of df1
df_PLS_filtered = df_PLS[df_PLS['date'] > max_time_corp]

# Concatenate
df_merged = pd.concat([df_corp, df_PLS_filtered], ignore_index=True)

# sort by date
df_merged = df_merged.sort_values('date').reset_index(drop=True)

In [11]:
#check that the span now is the entire period
#This dataset spans the following time period

min_time_merged = df_merged["date"].min()
max_time_merged = df_merged["date"].max()

print(f"The merged datasets has coverage from {min_time_merged.date()} to {max_time_merged.date()}")

The merged datasets has coverage from 1997-10-07 to 2022-10-06


# Now for sentence segmentation?

Implementatino of the DaCy pipeline. Due to incompatibility with newer python version. python version 3.10.19 used.

*Personal note: use env: dacy_env*


In [13]:
#import packages for sentence segmentation

import spacy
import dacy

In [ ]:
for model in dacy.models():
    print(model)

    

da_dacy_small_trf-0.2.0
da_dacy_medium_trf-0.2.0
da_dacy_large_trf-0.2.0
small
medium
large
da_dacy_small_ner_fine_grained-0.1.0
da_dacy_medium_ner_fine_grained-0.1.0
da_dacy_large_ner_fine_grained-0.1.0


In [11]:
# Specify model to be used

nlp = dacy.load("large") # Or da_dacy_large_trf-0.2.0

print(nlp.meta["name"]) # shoudl say the above name



ERROR: Invalid wheel filename (invalid version): 'da_dacy_large_trf-any-py3-none-any'


CalledProcessError: Command '['/work/Bachelor_my_workspace/miniconda3/envs/dacy_env/bin/python', '-m', 'pip', 'install', 'da_dacy_large_trf @ https://huggingface.co/chcaa/da_dacy_large_trf/resolve/963232f378190476503a1bfc35b520cb142e9e41/da_dacy_large_trf-any-py3-none-any.whl', '--no-deps']' returned non-zero exit status 1.

In [ ]:
#Setup example
doc = nlp(
    "Sætnings segmentering er en vigtig del af sprogprocessering - Det kan bl.a. benyttes til at opdele lange tekster i mindre bidder uden at miste meningen i hvert sætning."
)

for sent in doc.sents:
    print(sent)

In [ ]:
# lopp trough all rows of the dataset